# 2. Point Estimation: MLE and Method of Moments

Point estimation consists of finding the "best" single value for an unknown parameter. This notebook covers:
- Method of Moments (MoM)
- Maximum Likelihood Estimation (MLE)
- Log-likelihood visualization
- Fisher Information and asymptotic variance
- MLE for different distributions

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

%matplotlib inline

## 2.1 Exponential Distribution: MoM and MLE

For $X \sim \text{Exp}(\lambda)$, we have $E[X] = 1/\lambda$. Both the **Method of Moments** and **MLE** give $\hat{\lambda} = 1/\bar{X}$ (they coincide for the exponential family).

In [ ]:
data = np.array([1200, 1350, 980, 1500, 1100, 1450, 1300, 1050, 1400, 1250])

lambda_mom = 1 / np.mean(data)
lambda_mle = 1 / np.mean(data)
print(f"MoM: lambda_hat = {lambda_mom:.6f}")
print(f"MLE: lambda_hat = {lambda_mle:.6f}")

## 2.2 MLE for the Normal Distribution

For $X \sim \mathcal{N}(\mu, \sigma^2)$:
- $\hat{\mu}_{\text{MLE}} = \bar{X}$
- $\hat{\sigma}^2_{\text{MLE}} = \frac{1}{n}\sum(X_i - \bar{X})^2$ (divides by $n$, not $n-1$)

The MLE for $\sigma^2$ is **biased**; the unbiased estimator divides by $n-1$.

In [ ]:
mu_mle = np.mean(data)
sigma2_mle = np.var(data, ddof=0)  # MLE (biased)
sigma2_unbiased = np.var(data, ddof=1)  # Unbiased

print(f"mu_hat         = {mu_mle:.2f}")
print(f"sigma^2 (MLE)  = {sigma2_mle:.2f}")
print(f"sigma^2 (unbiased) = {sigma2_unbiased:.2f}")

## 2.3 Visualizing the Log-Likelihood

The log-likelihood function $\ell(\theta) = \sum_{i=1}^n \log f(x_i; \theta)$ is maximized at the MLE. Visualizing it shows how "peaked" the estimation is.

In [ ]:
lambdas = np.linspace(0.0005, 0.002, 200)
log_lik = np.array([np.sum(stats.expon.logpdf(data, scale=1/lam))
                    for lam in lambdas])

plt.figure(figsize=(8, 5))
plt.plot(lambdas, log_lik, 'b-', lw=2)
plt.axvline(lambda_mle, color='r', ls='--', label=f'MLE = {lambda_mle:.5f}')
plt.xlabel(r'$\lambda$')
plt.ylabel(r'$\ell(\lambda)$')
plt.title('Exponential Log-Likelihood')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2.4 Fisher Information

The **Fisher Information** $I(\theta) = -E\left[\frac{\partial^2 \ell}{\partial \theta^2}\right]$ determines the precision of the MLE. The asymptotic variance of $\hat{\theta}_{\text{MLE}}$ is $1/(n \cdot I(\theta))$.

In [ ]:
n = len(data)
I_fisher = 1 / lambda_mle**2  # For Exp(lambda)
var_asymp = 1 / (n * I_fisher)

print(f"Fisher Information I(lambda) = {I_fisher:.4f}")
print(f"Asymptotic variance: {var_asymp:.8f}")
print(f"95% asymptotic CI: [{lambda_mle - 1.96*np.sqrt(var_asymp):.5f}, "
      f"{lambda_mle + 1.96*np.sqrt(var_asymp):.5f}]")

## 2.5 MLE for Uniform U([0, theta])

For $X \sim U([0,\theta])$, the MLE is $\hat{\theta} = X_{(n)} = \max(X_1, \ldots, X_n)$.
This is a **biased** estimator: $E[X_{(n)}] = \frac{n}{n+1}\theta < \theta$.

In [ ]:
data_unif = np.array([0.3, 0.7, 0.5, 0.9, 0.2, 0.8, 0.6, 0.4, 0.95, 0.1])
theta_mle = np.max(data_unif)
n_u = len(data_unif)
theta_unbiased = (n_u + 1) / n_u * theta_mle

print(f"MLE:     theta_hat = {theta_mle}")
print(f"Unbiased: theta_hat = {theta_unbiased:.4f}")

## Key Takeaways

| Property | MLE |
|---|---|
| Consistency | Yes (converges to true value) |
| Asymptotic normality | $\hat{\theta} \approx \mathcal{N}(\theta, 1/(nI(\theta)))$ |
| Efficiency | Achieves the Cramer-Rao lower bound |
| Bias | May be biased in finite samples |